# Selección de características

Se utiliza `SelectKBest` con información mutua y el modelo LR para comparar conjuntos de 10, 15, 20 y 26 características. La comparación se realiza mediante validación cruzada estratificada de cinco particiones y utilizando únicamente el conjunto de entrenamiento.

## Importaciones

In [1]:
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, multilabel_confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## Lectura del conjunto de entrenamiento

In [2]:
def localizar_raiz_proyecto():
    """Localiza la carpeta principal del repositorio."""
    carpeta_actual = Path.cwd().resolve()

    if (carpeta_actual / "datos").exists():
        return carpeta_actual

    if (carpeta_actual.parent / "datos").exists():
        return carpeta_actual.parent

    raise FileNotFoundError(
        "No se encuentra la carpeta 'datos'. Abre el notebook desde el repositorio."
    )


RAIZ_PROYECTO = localizar_raiz_proyecto()
RUTA_TRAIN = RAIZ_PROYECTO / "datos" / "procesados" / "train.csv"


In [3]:
df_train = pd.read_csv(RUTA_TRAIN)

X_train = df_train.drop(columns=["clase"])
y_train = df_train["clase"]

clases = ["otro", "hipotiroidismo", "hipertiroidismo"]

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de y_train:", y_train.shape)
display(y_train.value_counts().to_frame())


Dimensiones de X_train: (7337, 26)
Dimensiones de y_train: (7337,)


,count
clase,
otro,6628
hipotiroidismo,527
hipertiroidismo,182


El conjunto de prueba no se utiliza durante la selección de características. Se mantiene reservado para la evaluación final de los modelos.

## Preprocesamiento

In [4]:
variables_numericas = [
    "edad", "TSH", "T3", "TT4", "T4U", "FTI"
]

variables_binarias = [
    "sexo",
    "tratamiento_con_tiroxina",
    "consulta_sobre_tiroxina",
    "medicacion_antitiroidea",
    "enfermo",
    "embarazada",
    "cirugia_tiroidea",
    "tratamiento_I131",
    "consulta_hipotiroidismo",
    "consulta_hipertiroidismo",
    "litio",
    "bocio",
    "tumor",
    "hipopituitarismo",
    "trastorno_psiquiatrico",
    "TSH_medida",
    "T3_medida",
    "TT4_medida",
    "T4U_medida",
    "FTI_medido",
]


In [5]:
procesador_numerico = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("normalizacion", StandardScaler()),
])

procesador_binario = Pipeline([
    ("imputacion", SimpleImputer(strategy="most_frequent")),
])

preprocesador = ColumnTransformer([
    ("numericas", procesador_numerico, variables_numericas),
    ("binarias", procesador_binario, variables_binarias),
])


## Pipeline de selección

In [6]:
informacion_mutua = partial(
    mutual_info_classif,
    random_state=42,
)

pipeline_seleccion = Pipeline([
    ("preprocesamiento", preprocesador),
    ("seleccion", SelectKBest(score_func=informacion_mutua)),
    ("modelo", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )),
])


## Métricas y validación cruzada

In [7]:
def calcular_especificidad_macro(y_real, y_predicha):
    """Calcula la media de la especificidad de las tres clases."""
    matrices = multilabel_confusion_matrix(
        y_real, y_predicha, labels=clases
    )

    especificidades = []
    for matriz in matrices:
        vn, fp, fn, vp = matriz.ravel()
        especificidad = vn / (vn + fp) if (vn + fp) > 0 else 0
        especificidades.append(especificidad)

    return np.mean(especificidades)


scorer_especificidad = make_scorer(calcular_especificidad_macro)

metricas = {
    "exactitud": "accuracy",
    "sensibilidad_macro": "recall_macro",
    "especificidad_macro": scorer_especificidad,
    "precision_macro": "precision_macro",
    "f1_macro": "f1_macro",
}


In [8]:
validacion = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

busqueda_k = GridSearchCV(
    estimator=pipeline_seleccion,
    param_grid={"seleccion__k": [10, 15, 20, 26]},
    scoring=metricas,
    refit="f1_macro",
    cv=validacion,
    n_jobs=-1,
    verbose=1,
)

busqueda_k.fit(X_train, y_train)


Fitting 5 folds for each of 4 candidates, totalling 20 fits


c:\Users\frana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocesamiento',
                                        ColumnTransformer(transformers=[('numericas',
                                                                         Pipeline(steps=[('imputacion',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('normalizacion',
                                                                                          StandardScaler())]),
                                                                         ['edad',
                                                                          'TSH',
                                                                          'T3',
                                                                          'TT4',
                                                                          'T4U',
                                                                          'FTI']),
                                                                        ('binarias',
                                                                         Pipeline(steps=[('imputacion',
                                                                                          Simp...
                                        LogisticRegression(class_weight='balanced',
                                                           max_iter=2000,
                                                           random_state=42))]),
             n_jobs=-1, param_grid={'seleccion__k': [10, 15, 20, 26]},
             refit='f1_macro',
             scoring={'especificidad_macro': make_scorer(calcular_especificidad_macro, response_method='predict'),
                      'exactitud': 'accuracy', 'f1_macro': 'f1_macro',
                      'precision_macro': 'precision_macro',
                      'sensibilidad_macro': 'recall_macro'},
             verbose=1)

## Resultados

In [9]:
resultados_k = pd.DataFrame({
    "Características": busqueda_k.cv_results_["param_seleccion__k"].astype(int),
    "Exactitud": busqueda_k.cv_results_["mean_test_exactitud"],
    "Sensibilidad macro": busqueda_k.cv_results_["mean_test_sensibilidad_macro"],
    "Especificidad macro": busqueda_k.cv_results_["mean_test_especificidad_macro"],
    "Precisión macro": busqueda_k.cv_results_["mean_test_precision_macro"],
    "F1-Score macro": busqueda_k.cv_results_["mean_test_f1_macro"],
})

resultados_k = resultados_k.sort_values("Características").reset_index(drop=True)
display(resultados_k.round(4))


,Características,Exactitud,Sensibilidad macro,Especificidad macro,Precisión macro,F1-Score macro
0,10,0.9185,0.9450,0.9636,0.6789,0.7512
1,15,0.9215,0.9401,0.9626,0.6830,0.7550
2,20,0.9220,0.9415,0.9627,0.6843,0.7565
3,26,0.9234,0.9414,0.9628,0.6871,0.7590


In [10]:
mejor_k = busqueda_k.best_params_["seleccion__k"]

print("Mejor número de características según F1 macro:", mejor_k)
print("Mejor F1 macro medio:", round(busqueda_k.best_score_, 4))


Mejor número de características según F1 macro: 26
Mejor F1 macro medio: 0.759


## Características seleccionadas

In [11]:
mejor_pipeline = busqueda_k.best_estimator_
selector = mejor_pipeline.named_steps["seleccion"]
preprocesador_ajustado = mejor_pipeline.named_steps["preprocesamiento"]

nombres_variables = preprocesador_ajustado.get_feature_names_out()
variables_seleccionadas = nombres_variables[selector.get_support()]

for variable in variables_seleccionadas:
    print(variable.split("__")[-1])


edad
TSH
T3
TT4
T4U
FTI
sexo
tratamiento_con_tiroxina
consulta_sobre_tiroxina
medicacion_antitiroidea
enfermo
embarazada
cirugia_tiroidea
tratamiento_I131
consulta_hipotiroidismo
consulta_hipertiroidismo
litio
bocio
tumor
hipopituitarismo
trastorno_psiquiatrico
TSH_medida
T3_medida
TT4_medida
T4U_medida
FTI_medido


## Conclusión

La selección se decide tomando como referencia el F1-Score macro. El mejor resultado se obtiene con las 26 características, se mantienen todas las variables en los entrenamientos posteriores.